# Lab 3.4 — Performance Evaluation: Runtime vs Mantle

**Duration:** ~7 min · **Level:** L300 · **Lab 3 of 4 — part 4/4**

We measure:

1. **TTFT (time-to-first-token)** — how long before the first visible byte.
2. **Tokens per second** — streaming throughput in the middle of the stream.
3. **End-to-end latency** for the full streamed completion.

This notebook is intentionally small — 10 samples per endpoint — so it
finishes in under a minute. For a real migration, increase `N_SAMPLES`
to 500+ per model × region and collect p50 / p95 / p99 per the playbook §8.


In [ ]:
import os, sys, time, statistics
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "src" / "common").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

import boto3
from src.common.mantle import openai_client, GPT_OSS_120B

runtime = boto3.client("bedrock-runtime")
mantle  = openai_client()

N_SAMPLES = 10
PROMPT = "Write a single 40-word paragraph about the AWS Nitro hypervisor."
MAX_OUT = 80


## 1. Benchmark helper — Mantle Chat Completions (streaming)

Returns `(ttft_seconds, tokens_per_second, total_seconds)`.


In [ ]:
def bench_mantle_chat(model: str) -> tuple[float, float, float]:
    t0 = time.time()
    ttft = None
    frags = 0
    stream = mantle.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": PROMPT}],
        max_tokens=MAX_OUT,
        stream=True,
        stream_options={"include_usage": True},
    )
    last = t0
    for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content:
            if ttft is None:
                ttft = time.time() - t0
            frags += 1
            last = time.time()
    total = time.time() - t0
    stream_dur = max(last - (t0 + (ttft or 0)), 1e-6)
    tps = frags / stream_dur
    return ttft or 0.0, tps, total


## 2. Benchmark helper — Runtime ConverseStream

The runtime path yields structured `contentBlockDelta` events. We count each
`text` delta as a "fragment" — it's apples-to-apples with Mantle's
`delta.content` fragments for ordering / count purposes.


In [ ]:
def bench_runtime_converse_stream(model: str) -> tuple[float, float, float]:
    t0 = time.time()
    ttft = None
    frags = 0
    resp = runtime.converse_stream(
        modelId=model,
        messages=[{"role": "user", "content": [{"text": PROMPT}]}],
        inferenceConfig={"maxTokens": MAX_OUT, "temperature": 0.2},
    )
    last = t0
    for event in resp["stream"]:
        if "contentBlockDelta" in event:
            delta = event["contentBlockDelta"].get("delta", {})
            if "text" in delta:
                if ttft is None:
                    ttft = time.time() - t0
                frags += 1
                last = time.time()
    total = time.time() - t0
    stream_dur = max(last - (t0 + (ttft or 0)), 1e-6)
    tps = frags / stream_dur
    return ttft or 0.0, tps, total


## 3. Run the samples

This takes ~30 seconds. If you're in a latency-sensitive environment, you
may want to run this cell a few times before capturing numbers (to let
connections warm up).


In [ ]:
def stats(name, samples):
    ttfts = [s[0] for s in samples if s[0] > 0]
    tpss  = [s[1] for s in samples if s[1] > 0]
    tot   = [s[2] for s in samples]

    def pct(xs, p):
        if not xs:
            return float("nan")
        if len(xs) == 1:
            return xs[0]
        return statistics.quantiles(xs, n=100)[p-1]

    print(f"{name:>30}  ttft p50={pct(ttfts,50):.2f}s  p95={pct(ttfts,95):.2f}s"
          f"  tps p50={pct(tpss,50):.1f}  total p50={pct(tot,50):.2f}s"
          f"  (n={len(samples)})")

mantle_samples = [bench_mantle_chat(GPT_OSS_120B) for _ in range(N_SAMPLES)]
stats("mantle chat_completions", mantle_samples)

# Try a runtime model that your account is very likely to have.
CANDIDATES = ["anthropic.claude-haiku-4-5",
              "anthropic.claude-3-5-haiku-20241022-v1:0"]
runtime_samples = None
for cand in CANDIDATES:
    try:
        runtime_samples = [bench_runtime_converse_stream(cand) for _ in range(N_SAMPLES)]
        stats(f"runtime {cand}", runtime_samples)
        break
    except Exception as e:
        print(f"  skipping {cand}: {e.__class__.__name__}")

if runtime_samples is None:
    print("⚠️  No runtime model available — Mantle-only numbers above.")


## 4. Interpreting the numbers

- **TTFT dominates perceived latency.** A 1.5 s TTFT on a chat UX feels
  sluggish even if tokens/sec is fast afterwards.
- **Cold starts matter.** First call to any region is slower; most benchmarks
  exclude the first 2-3 samples.
- **Apples-to-apples requires the same model family.** Different model sizes
  produce different tokens/sec — don't compare a 20B to a 120B.
- **TPS drift over stream length.** Some models are front-loaded (fast first,
  slower later). Long completions should be sampled at p50 across sentences.

For a statistically meaningful comparison per the playbook §8:

- ≥ 500 samples per (model, region, endpoint) tuple.
- Capture p50 / p95 / p99 TTFT + total.
- Separate samples by hour-of-day (diurnal load matters on shared fleets).
- Include throttle / error rates next to latency.


## 5. A simple plot (optional)

If matplotlib is installed, plot the TTFT distributions.


In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 3.5))
    bins = 10
    ax.hist([s[0] for s in mantle_samples], bins=bins, alpha=0.6, label="Mantle CC")
    if runtime_samples:
        ax.hist([s[0] for s in runtime_samples], bins=bins, alpha=0.6, label="Runtime Converse")
    ax.set_xlabel("TTFT (s)")
    ax.set_ylabel("count")
    ax.set_title(f"TTFT distribution (n={N_SAMPLES})")
    ax.legend()
    fig.tight_layout()
    plt.show()
except ModuleNotFoundError:
    print("install matplotlib to plot: pip install matplotlib")
